# 03b — ARIMA & LSTM Models

06_ARIMA_LSTM.py — Classical ARIMA Baseline & LSTM Deep Learning Model
TFM: Demand Forecasting for Inventory Management Using Machine Learning
Author: Ona Mas i Serra

Adds two models to the comparison:
  1. ARIMA (auto_arima via pmdarima) — classical time series baseline
  2. LSTM (PyTorch) — deep learning sequential model

Reads: output/processed/features.parquet, output/models/predictions_last_fold.parquet
Outputs: output/models/arima_lstm_results.csv, plots


In [ ]:
%matplotlib inline


import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
import time
import json
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from style_config import (apply_style, PRIMARY, ACCENT, SECONDARY, ALERT,
                          get_model_color, get_model_colors)

warnings.filterwarnings("ignore")
apply_style()

# ── Configuration ──────────────────────────────────────────────────
BASE_DIR = Path(".")
FEATURES_PATH = BASE_DIR / "output" / "processed" / "features.parquet"
PREDS_PATH = BASE_DIR / "output" / "models" / "predictions_last_fold.parquet"
MODEL_DIR = BASE_DIR / "output" / "models"
PLOT_DIR = BASE_DIR / "output" / "models" / "plots"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "udsVenta"
EXCLUDE = ["producto", "idSecuencia", "fecha", "day_name", TARGET,
           "prod_mean_sales", "prod_std_sales", "prod_median_sales"]

# ARIMA config
N_ARIMA_PRODUCTS = None  # None = all products
SKIP_ARIMA = False

# LSTM config
LSTM_HIDDEN = 32
LSTM_SEQ_LEN = 14
LSTM_EPOCHS = 10
LSTM_BATCH = 512
LSTM_LR = 0.003
N_LSTM_TRAIN_PRODUCTS = 200  # sample for training speed

def save_fig(name):
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{name}.png", bbox_inches="tight", dpi=120)
    plt.close()
    print(f"   → Saved {name}.png")

def calc_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    errors = y_true - y_pred
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true > 0
    mape = np.mean(np.abs(errors[mask] / y_true[mask])) * 100 if mask.sum() > 0 else np.nan
    sigma_e = np.std(errors)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "sigma_e": sigma_e}


# ══════════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ══════════════════════════════════════════════════════════════════
print("=" * 60)
print("1. LOADING DATA")
print("=" * 60)

df = pd.read_parquet(FEATURES_PATH)
df = df.sort_values(["producto", "fecha"]).reset_index(drop=True)
feature_cols = [c for c in df.columns if c not in EXCLUDE]
all_products = df["producto"].unique()

all_dates = sorted(df["fecha"].unique())
cutoff_idx = len(all_dates) - 30
cutoff_date = all_dates[cutoff_idx]
test_start = all_dates[cutoff_idx]
test_end = all_dates[-1]

print(f"Shape: {df.shape}, Features: {len(feature_cols)}")
print(f"Test: {pd.Timestamp(test_start).date()} → {pd.Timestamp(test_end).date()}")


# ══════════════════════════════════════════════════════════════════
# 2. ARIMA BASELINE
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("2. ARIMA BASELINE")
print("=" * 60)

if SKIP_ARIMA:
    # Use cached results
    arima_mean = {
        "model": "ARIMA",
        "MAE": 1.443, "RMSE": 2.220, "MAPE": 59.8, "sigma_e": 2.219,
        "note": "cached from previous run"
    }
    print(f"ARIMA (cached): MAE={arima_mean['MAE']:.3f}  RMSE={arima_mean['RMSE']:.3f}")
else:
    import pmdarima as pm

    arima_products = all_products if N_ARIMA_PRODUCTS is None else np.random.choice(
        all_products, size=N_ARIMA_PRODUCTS, replace=False)
    n_arima = len(arima_products)
    print(f"Running ARIMA on {n_arima} products...")

    arima_preds_list = []
    arima_errors = 0
    t0 = time.time()

    for i, prod in enumerate(arima_products):
        prod_df = df[df["producto"] == prod].sort_values("fecha")
        train_series = prod_df[prod_df["fecha"] < cutoff_date][TARGET].values
        test_series = prod_df[(prod_df["fecha"] >= test_start) & (prod_df["fecha"] <= test_end)]

        if len(train_series) < 30 or len(test_series) == 0:
            arima_errors += 1
            continue

        try:
            model = pm.auto_arima(train_series, seasonal=True, m=7,
                                   suppress_warnings=True, error_action="ignore",
                                   max_p=3, max_q=3, max_P=2, max_Q=2, max_d=2, max_D=1,
                                   stepwise=True, n_fits=20)
            fc = model.predict(n_periods=len(test_series))
            fc = np.clip(fc, 0, None)

            for j, (_, row) in enumerate(test_series.iterrows()):
                arima_preds_list.append({
                    "producto": prod,
                    "fecha": row["fecha"],
                    TARGET: row[TARGET],
                    "ARIMA": fc[j]
                })
        except Exception:
            arima_errors += 1

        if (i + 1) % 50 == 0 or i == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed * 60
            remaining = (n_arima - i - 1) / rate if rate > 0 else 0
            print(f"  [{i+1}/{n_arima}] {elapsed:.0f}s elapsed, ~{remaining:.0f}min remaining, {arima_errors} errors")

    elapsed = time.time() - t0
    print(f"\nARIMA complete: {elapsed:.0f}s ({elapsed/60:.1f}min), {arima_errors} errors out of {n_arima}")

    arima_pred_df = pd.DataFrame(arima_preds_list)
    arima_m = calc_metrics(arima_pred_df[TARGET], arima_pred_df["ARIMA"])
    arima_mean = {"model": "ARIMA", **arima_m, "note": f"all {n_arima} products"}
    print(f"  ARIMA: MAE={arima_m['MAE']:.3f}  RMSE={arima_m['RMSE']:.3f}  "
          f"MAPE={arima_m['MAPE']:.1f}%  σ_e={arima_m['sigma_e']:.3f}")

    # Save ARIMA predictions
    arima_pred_df.to_parquet(MODEL_DIR / "arima_predictions.parquet", index=False)
    print(f"✓ Saved arima_predictions.parquet ({len(arima_pred_df):,} rows)")


# ══════════════════════════════════════════════════════════════════
# 3. LSTM MODEL (PyTorch)
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("3. LSTM MODEL (PyTorch)")
print("=" * 60)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cpu")
print(f"Device: {device}")

# ── 3.1 Prepare data ──
train_df = df[df["fecha"] < cutoff_date].copy()
test_df = df[(df["fecha"] >= test_start) & (df["fecha"] <= test_end)].copy()

# Sample products for training
np.random.seed(42)
lstm_train_prods = np.random.choice(all_products, size=N_LSTM_TRAIN_PRODUCTS, replace=False)
train_sampled = train_df[train_df["producto"].isin(lstm_train_prods)]

scaler = StandardScaler()
scaler.fit(train_sampled[feature_cols])

# ── 3.2 Build sequences ──
print(f"Building sequences (seq_len={LSTM_SEQ_LEN})...")

def make_sequences(product_df, scaler, seq_len, date_filter=None):
    """Create (X, y, meta) sequences grouped by product."""
    X_all, y_all, meta = [], [], []
    for prod, grp in product_df.groupby("producto"):
        grp = grp.sort_values("fecha")
        feats = scaler.transform(grp[feature_cols])
        targets = grp[TARGET].values
        fechas = grp["fecha"].values
        for i in range(seq_len, len(feats)):
            if date_filter is not None and fechas[i] < date_filter:
                continue
            X_all.append(feats[i - seq_len:i])
            y_all.append(targets[i])
            meta.append({"producto": prod, "fecha": fechas[i]})
    return np.array(X_all), np.array(y_all), meta

X_train, y_train, _ = make_sequences(train_sampled, scaler, LSTM_SEQ_LEN)
print(f"Train sequences: {X_train.shape[0]:,}")

# For test: combine tail of train + test per product so sequences span the boundary
combined = pd.concat([train_df, test_df]).sort_values(["producto", "fecha"])
X_test, y_test, meta_test = make_sequences(combined, scaler, LSTM_SEQ_LEN, date_filter=np.datetime64(test_start))
print(f"Test sequences: {X_test.shape[0]:,}")


# ── 3.3 PyTorch model ──
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMModel(nn.Module):
    def __init__(self, n_features, hidden):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=1, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


# ── 3.4 Train ──
print(f"\nTraining LSTM (hidden={LSTM_HIDDEN}, epochs={LSTM_EPOCHS}, batch={LSTM_BATCH})...")

train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=LSTM_BATCH, shuffle=True)
model = LSTMModel(len(feature_cols), LSTM_HIDDEN).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_LR)
criterion = nn.MSELoss()

t0 = time.time()
for epoch in range(LSTM_EPOCHS):
    model.train()
    total_loss, n = 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n += 1
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1}/{LSTM_EPOCHS}: loss={total_loss/n:.4f}")

print(f"Training: {time.time()-t0:.0f}s")


# ── 3.5 Evaluate ──
print("\nEvaluating LSTM...")
model.eval()
test_loader = DataLoader(SeqDataset(X_test, y_test), batch_size=LSTM_BATCH, shuffle=False)
preds_list = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds_list.append(model(xb.to(device)).cpu().numpy())

lstm_preds = np.clip(np.concatenate(preds_list), 0, None)
lstm_m = calc_metrics(y_test, lstm_preds)
print(f"  LSTM: MAE={lstm_m['MAE']:.3f}  RMSE={lstm_m['RMSE']:.3f}  "
      f"MAPE={lstm_m['MAPE']:.1f}%  σ_e={lstm_m['sigma_e']:.3f}")


# ══════════════════════════════════════════════════════════════════
# 4. COMBINED COMPARISON
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("4. COMBINED COMPARISON")
print("=" * 60)

# Build LSTM prediction df
lstm_df = pd.DataFrame(meta_test)
lstm_df["LSTM"] = lstm_preds

# Merge with existing predictions
existing = pd.read_parquet(PREDS_PATH)
merged = existing.merge(lstm_df[["producto", "fecha", "LSTM"]], on=["producto", "fecha"], how="inner")
print(f"Merged: {len(merged):,} rows, {merged['producto'].nunique()} products")

all_models = ["Naive_lag7", "RandomForest", "XGBoost", "LightGBM", "LSTM"]
print(f"\n── All Models (test period) ──")
rows = []
for m in all_models:
    met = calc_metrics(merged[TARGET], merged[m])
    met["model"] = m
    rows.append(met)
    print(f"  {m:15s}: MAE={met['MAE']:.3f}  RMSE={met['RMSE']:.3f}  "
          f"MAPE={met['MAPE']:.1f}%  σ_e={met['sigma_e']:.3f}")

print(f"\n  {'ARIMA*':15s}: MAE={arima_mean['MAE']:.3f}  RMSE={arima_mean['RMSE']:.3f}  "
      f"MAPE={arima_mean['MAPE']:.1f}%  σ_e={arima_mean['sigma_e']:.3f}  "
      f"(50 sampled products)")

# Promo vs Non-Promo
print(f"\n── LSTM: Promo vs Non-Promo ──")
for label, mask in [("Non-Promo", merged["en_promo"] == 0), ("Promo", merged["en_promo"] == 1)]:
    sub = merged[mask]
    if len(sub) == 0:
        continue
    met = calc_metrics(sub[TARGET], sub["LSTM"])
    print(f"  {label:10s}: MAE={met['MAE']:.3f}  RMSE={met['RMSE']:.3f}  σ_e={met['sigma_e']:.3f}")


# ══════════════════════════════════════════════════════════════════
# 5. SAVE & PLOT
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("5. SAVING RESULTS")
print("=" * 60)

summary = pd.DataFrame(rows + [arima_mean])
summary.to_csv(MODEL_DIR / "arima_lstm_results.csv", index=False)
merged.to_parquet(MODEL_DIR / "predictions_all_models.parquet", index=False)
print("✓ Saved arima_lstm_results.csv & predictions_all_models.parquet")

# Plot: all models comparison
fig, ax = plt.subplots(figsize=(10, 6))
labels = all_models + ["ARIMA*"]
rmses = [calc_metrics(merged[TARGET], merged[m])["RMSE"] for m in all_models] + [arima_mean["RMSE"]]
colors = [get_model_color(l) for l in labels]
ax.barh(labels, rmses, color=colors, edgecolor="white")
ax.set_xlabel("RMSE")
ax.set_title("All Models — RMSE Comparison")
save_fig("24_all_models_comparison")

# Plot: LSTM actual vs predicted
np.random.seed(42)
idx = np.random.choice(len(merged), min(5000, len(merged)), replace=False)
sample = merged.iloc[idx]
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(sample[TARGET], sample["LSTM"], alpha=0.2, s=5, color=get_model_color("LSTM"))
lim = max(sample[TARGET].max(), sample["LSTM"].max())
ax.plot([0, lim], [0, lim], color=ALERT, linestyle="--", linewidth=1, label="Perfect prediction")
ax.set_xlabel("Actual Sales")
ax.set_ylabel("Predicted Sales (LSTM)")
ax.set_title("Actual vs Predicted — LSTM")
ax.legend()
save_fig("25_actual_vs_predicted_lstm")

print("\n" + "=" * 60)
print("ARIMA + LSTM COMPLETE")
print("=" * 60)

